# Module 1 Final: PhoBERT Training + Best Model + EN/VI Inference

This merged notebook keeps the improved training pipeline from the new notebook and restores bilingual display/inference support from the old notebook.

In [ ]:
!pip install -q --upgrade transformers>=4.41.0
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q --upgrade scikit-learn
!pip install -q --upgrade seaborn matplotlib


## Run on Google Colab


In [ ]:

import os
import json
import random
import re
import warnings
from datetime import datetime
from pathlib import Path
from collections import Counter


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm


import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler


from transformers import (
    AutoTokenizer, 
    AutoModel, 
    get_linear_schedule_with_warmup,
    get_cosine_schedule_with_warmup
)


from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    f1_score,
    recall_score
)
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings('ignore')
sns.set_palette("husl")
plt.style.use('default')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from google.colab import files

train_uploaded = files.upload()
train_filename = list(train_uploaded.keys())[0]
print(f" Uploaded: {train_filename}")
val_uploaded = files.upload()
val_filename = list(val_uploaded.keys())[0]
print(f" Uploaded: {val_filename}")
test_uploaded = files.upload()
test_filename = list(test_uploaded.keys())[0]
print(f" Uploaded: {test_filename}")


## Hyperparameter Configuration 

(này tao để AI tool chứ cũng không biết phải tune như nào nhưng mà có double check + tham khảo nhiều con khác nhau lẫn nguồn trên mạng )

In [ ]:
class Config:
    """Optimized configuration for balanced dataset"""
    
    # Paths
    TRAIN_PATH = train_filename
    VAL_PATH = val_filename
    TEST_PATH = test_filename
    OUTPUT_DIR = 'outputs_v2_balanced'
    MODEL_SAVE_PATH = 'best_model_v2.pt'
    
    # Model
    MODEL_NAME = 'vinai/phobert-base'
    NUM_LABELS = 4
    MAX_LENGTH = 64  # Text avg ~10 words
    
    # Architecture - Optimized for balanced data
    HIDDEN_DIMS = [768, 384, 128]  # Multi-layer classifier
    DROPOUT = 0.3
    USE_BATCH_NORM = True
    
   
    BATCH_SIZE = 24  
    NUM_EPOCHS = 20  
    LEARNING_RATE = 3e-5  
    WEIGHT_DECAY = 0.01
    WARMUP_RATIO = 0.1  
    MAX_GRAD_NORM = 1.0
    
    # Loss function - LIGHTER approach
    USE_FOCAL_LOSS = False  # Standard CE is enough!
    USE_CLASS_WEIGHTS = True  # Mild weighting only
    LABEL_SMOOTHING = 0.05  
    # Data augmentation - OPTIONAL (data already good)
    USE_AUGMENTATION = True  # Light boost only
    AUGMENT_MINORITY_ONLY = True  # Only Label 4
    AUG_FACTOR = 1.3  # Gentle augmentation (not 2x)
    
    # Sampling - STANDARD (no aggressive weighting)
    USE_WEIGHTED_SAMPLING = False  # Not needed!
    
    # Early stopping
    PATIENCE = 4
    MIN_DELTA = 0.001
    
    # Scheduler
    SCHEDULER_TYPE = 'cosine'
    
    # Random seed
    SEED = 42
    
    # Device
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Class names
    LABEL_NAMES_VI = {
        0: 'Không có triệu chứng',
        1: 'Nhẹ',
        2: 'Vừa',
        3: 'Nặng'
    }
    LABEL_NAMES_EN = {
        0: 'None',
        1: 'Mild',
        2: 'Moderate',
        3: 'Severe'
    }
    DISPLAY_LANG = 'vi'  # 'vi' or 'en'

config = Config()
Path(config.OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f"  Device: {config.DEVICE}")
print(f"  Batch size: {config.BATCH_SIZE}")
print(f"  Epochs: {config.NUM_EPOCHS}")
print(f"  Learning rate: {config.LEARNING_RATE}")
print(f"  Max length: {config.MAX_LENGTH}")
print(f"  Focal Loss: {config.USE_FOCAL_LOSS}")
print(f"  Class Weights: {config.USE_CLASS_WEIGHTS}")
print(f"  Augmentation: {config.USE_AUGMENTATION}")

LABEL_MAP_VI = {'None': 'Không có triệu chứng', 'Mild': 'Nhẹ', 'Moderate': 'Vừa', 'Severe': 'Nặng'}
LABEL_MAP_EN = {0: 'None', 1: 'Mild', 2: 'Moderate', 3: 'Severe'}

def get_display_label(label_idx=None, label_text=None, lang='vi'):
    if label_idx is not None:
        return config.LABEL_NAMES_EN[label_idx] if lang == 'en' else config.LABEL_NAMES_VI[label_idx]
    if label_text is not None:
        return label_text if lang == 'en' else LABEL_MAP_VI.get(label_text, label_text)
    return ''


In [ ]:
def set_seed(seed):
    """Set random seed for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(config.SEED)

In [ ]:
# Load data - Keep separate!
print(" Loading data...")
train_df = pd.read_csv(config.TRAIN_PATH)
val_df = pd.read_csv(config.VAL_PATH)
test_df = pd.read_csv(config.TEST_PATH)

print(f"\n Data loaded:")
print(f"  Train: {len(train_df):,} samples")
print(f"  Val:   {len(val_df):,} samples")
print(f"  Test:  {len(test_df):,} samples")
print(f"  Total: {len(train_df) + len(val_df) + len(test_df):,} samples")

print(f"\nColumns: {list(train_df.columns)}")
train_df.head()

## Label Distribution and Class Balance


In [ ]:
# Label distribution analysis
print("="*70)
print(" LABEL DISTRIBUTION - IMPROVED! ")
print("="*70)

for split_name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    print(f"\n{split_name}:")
    label_counts = df['label'].value_counts().sort_index()
    for label, count in label_counts.items():
        percentage = (count / len(df)) * 100
        print(f"  Label {label}: {count:4,} ({percentage:5.1f}%)")

# Calculate imbalance
train_dist = train_df['label'].value_counts().sort_index()
imbalance_ratio = train_dist.max() / train_dist.min()
print(f"\n  Imbalance Ratio: {imbalance_ratio:.2f}:1")
if imbalance_ratio < 2.5:
    print("    GOOD - Mild imbalance (much better than 4.5:1!)")
elif imbalance_ratio < 4:
    print("     MODERATE - Manageable")
else:
    print("    SEVERE")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, df) in enumerate([('Train', train_df), ('Val', val_df), ('Test', test_df)]):
    label_counts = df['label'].value_counts().sort_index()
    colors = ['#ff9999', '#66b3ff', '#99ff99', '#ffcc99']
    axes[idx].bar(label_counts.index, label_counts.values, 
                  color=colors, edgecolor='black', linewidth=1.5)
    axes[idx].set_title(f'{name} Set - BALANCED \n({len(df):,} samples)', 
                        fontsize=13, fontweight='bold')
    axes[idx].set_xlabel('Label', fontsize=11)
    axes[idx].set_ylabel('Count', fontsize=11)
    axes[idx].grid(axis='y', alpha=0.3)
    
    for i, v in enumerate(label_counts.values):
        axes[idx].text(label_counts.index[i], v + 30, str(v), 
                       ha='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig(f'{config.OUTPUT_DIR}/label_distribution.png', dpi=300, bbox_inches='tight')
plt.show()


## Text Preprocessing


In [ ]:
def preprocess_text(text):
    """
    Clean text - Keep punctuation for emotion
    """
    if pd.isna(text):
        return ""
    
    text = str(text).strip()
    text = ' '.join(text.split())  # Normalize whitespace
    text = re.sub(r'http\S+|www\S+|\S+@\S+', '', text)  # Remove URLs/emails
    text = re.sub(r'([!?.]){4,}', r'\1\1\1', text)  # Max 3 repeating punctuation
    
    return text.strip()

# Test
test_cases = [
    "MẮT TÔI RẤT ĐAU!!!",
    "Mắt khô, cần nhắm mắt",
    "Không có vấn đề."
]

print(" Preprocessing Test:")
print("="*70)
for text in test_cases:
    cleaned = preprocess_text(text)
    print(f"Original: {text}")
    print(f"Cleaned:  {cleaned}")
    print("-" * 70)

In [ ]:
# Apply preprocessing SEPARATELY
print(" Applying preprocessing...")

train_df['text_clean'] = train_df['description'].apply(preprocess_text)
val_df['text_clean'] = val_df['description'].apply(preprocess_text)
test_df['text_clean'] = test_df['description'].apply(preprocess_text)

# Convert labels 1-4 to 0-3
train_df['label_id'] = train_df['label'] - 1
val_df['label_id'] = val_df['label'] - 1
test_df['label_id'] = test_df['label'] - 1

# Remove empty
for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    empty_count = (df['text_clean'] == '').sum()
    if empty_count > 0:
        print(f"  Removing {empty_count} empty texts from {name}")
        if name == 'train':
            train_df = train_df[train_df['text_clean'] != ''].reset_index(drop=True)
        elif name == 'val':
            val_df = val_df[val_df['text_clean'] != ''].reset_index(drop=True)
        else:
            test_df = test_df[test_df['text_clean'] != ''].reset_index(drop=True)

print(f"\n Preprocessing completed:")
print(f"  Train: {len(train_df):,} samples")
print(f"  Val:   {len(val_df):,} samples")
print(f"  Test:  {len(test_df):,} samples")

## Light Augmentation for Minority Class

In [ ]:
def light_augment(text, n=1):
    """Light augmentation for minority class"""
    augmented = []
    words = text.split()
    
    for _ in range(n):
        if len(words) > 3:
            # Shuffle 1-2 words only
            shuffled = words.copy()
            if random.random() > 0.5:
                idx1, idx2 = random.sample(range(len(words)), 2)
                shuffled[idx1], shuffled[idx2] = shuffled[idx2], shuffled[idx1]
            augmented.append(' '.join(shuffled))
        else:
            augmented.append(text)
    
    return augmented


if config.USE_AUGMENTATION:
    print(" Applying LIGHT augmentation...")
    
    label_counts = train_df['label_id'].value_counts().sort_index()
    majority_count = label_counts.max()
    
    augmented_rows = []
    
    # Only augment Label 4 (minority)
    label_id = 3  # Label 4
    samples = train_df[train_df['label_id'] == label_id]
    current_count = len(samples)
    target_count = int(current_count * config.AUG_FACTOR)  # Just 30% boost
    needed = target_count - current_count
    
    if needed > 0:
        print(f"\n  Label 4: {current_count} → {target_count} (+{needed} samples)")
        
        to_augment = samples.sample(n=needed, replace=True)
        
        for _, row in to_augment.iterrows():
            aug_text = light_augment(row['text_clean'], n=1)[0]
            augmented_rows.append({
                'description': row['description'],
                'text_clean': aug_text,
                'label': row['label'],
                'label_id': row['label_id']
            })
    
    if augmented_rows:
        aug_df = pd.DataFrame(augmented_rows)
        train_df = pd.concat([train_df, aug_df], ignore_index=True)
        train_df = train_df.sample(frac=1, random_state=config.SEED).reset_index(drop=True)
        
        print(f"  Total training: {len(train_df):,} (was {len(train_df)-len(aug_df):,})")
        print(f"  Added: {len(aug_df):,} samples")
        
        print("\n  New distribution:")
        for label_id in range(4):
            count = (train_df['label_id'] == label_id).sum()
            pct = count / len(train_df) * 100
            print(f"    Label {label_id+1}: {count:,} ({pct:.1f}%)")

In [ ]:
# Load tokenizer
print(" Loading PhoBERT tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)
print(f" Tokenizer loaded: {config.MODEL_NAME}")

# Test
test_text = "Mắt tôi rất đau!!!"
test_encoding = tokenizer(
    test_text,
    add_special_tokens=True,
    max_length=config.MAX_LENGTH,
    padding='max_length',
    truncation=True,
    return_tensors='pt'
)

print(f"\n Test tokenization:")
print(f"  Text: {test_text}")
print(f"  Tokens: {tokenizer.convert_ids_to_tokens(test_encoding['input_ids'][0][:15])}")
print(f"  Shape: {test_encoding['input_ids'].shape}")

In [ ]:
class EyeStrainDataset(Dataset):
    """Dataset for balanced data"""
    
    def __init__(self, texts, labels, tokenizer, max_length=64):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Create datasets
train_dataset = EyeStrainDataset(
    texts=train_df['text_clean'].tolist(),
    labels=train_df['label_id'].tolist(),
    tokenizer=tokenizer,
    max_length=config.MAX_LENGTH
)

val_dataset = EyeStrainDataset(
    texts=val_df['text_clean'].tolist(),
    labels=val_df['label_id'].tolist(),
    tokenizer=tokenizer,
    max_length=config.MAX_LENGTH
)

test_dataset = EyeStrainDataset(
    texts=test_df['text_clean'].tolist(),
    labels=test_df['label_id'].tolist(),
    tokenizer=tokenizer,
    max_length=config.MAX_LENGTH
)

print(f" Datasets created:")
print(f"  Train: {len(train_dataset):,} samples")
print(f"  Val:   {len(val_dataset):,} samples")
print(f"  Test:  {len(test_dataset):,} samples")

In [ ]:
# Create DataLoaders - STANDARD sampling (no weighted sampling needed!)
train_loader = DataLoader(
    train_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=True  # Standard shuffle
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False
)

print(f" DataLoaders created:")
print(f"  Train: {len(train_loader)} batches")
print(f"  Val:   {len(val_loader)} batches")
print(f"  Test:  {len(test_loader)} batches")

In [ ]:
class PhoBERTClassifier(nn.Module):
    """PhoBERT classifier with multi-layer head"""
    
    def __init__(self, num_labels=4, hidden_dims=[768, 384, 128], 
                 dropout=0.3, use_batch_norm=True):
        super(PhoBERTClassifier, self).__init__()
        
        self.phobert = AutoModel.from_pretrained(config.MODEL_NAME)
        
        # Multi-layer classifier
        layers = []
        
        for i in range(len(hidden_dims) - 1):
            layers.append(nn.Linear(hidden_dims[i], hidden_dims[i+1]))
            
            if use_batch_norm:
                layers.append(nn.BatchNorm1d(hidden_dims[i+1]))
            
            layers.append(nn.GELU())
            layers.append(nn.Dropout(dropout if i == 0 else dropout/2))
        
        layers.append(nn.Linear(hidden_dims[-1], num_labels))
        
        self.classifier = nn.Sequential(*layers)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.phobert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        pooled_output = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(pooled_output)
        
        return logits

# Initialize
print("  Initializing model...")
model = PhoBERTClassifier(
    num_labels=config.NUM_LABELS,
    hidden_dims=config.HIDDEN_DIMS,
    dropout=config.DROPOUT,
    use_batch_norm=config.USE_BATCH_NORM
)

model = model.to(config.DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n Model initialized:")
print(f"  Architecture: {config.HIDDEN_DIMS} → {config.NUM_LABELS}")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable: {trainable_params:,}")
print(f"  Device: {config.DEVICE}")

In [ ]:
# Compute class weights (LIGHT only)
print("  Computing class weights...")
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df['label_id']),
    y=train_df['label_id']
)

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(config.DEVICE)

print("\nClass weights (lighter than V1):")
for i, weight in enumerate(class_weights):
    print(f"  Label {i+1} ({config.LABEL_NAMES_VI[i]}): {weight:.3f}")

# Loss function - SIMPLE CrossEntropy (no Focal Loss needed!)
if config.USE_CLASS_WEIGHTS:
    criterion = nn.CrossEntropyLoss(
        weight=class_weights_tensor,
        label_smoothing=config.LABEL_SMOOTHING
    )
    print(f"\n Using CrossEntropyLoss with class weights + label smoothing ({config.LABEL_SMOOTHING})")
    print("   (No Focal Loss needed - data is balanced!)")
else:
    criterion = nn.CrossEntropyLoss(label_smoothing=config.LABEL_SMOOTHING)
    print(f"\n Using standard CrossEntropyLoss + label smoothing ({config.LABEL_SMOOTHING})")

In [ ]:
# Optimizer with differential learning rates
bert_params = list(model.phobert.named_parameters())
classifier_params = list(model.classifier.named_parameters())

optimizer_grouped_parameters = [
    {
        'params': [p for n, p in bert_params],
        'lr': config.LEARNING_RATE,
        'weight_decay': config.WEIGHT_DECAY
    },
    {
        'params': [p for n, p in classifier_params],
        'lr': config.LEARNING_RATE * 10,
        'weight_decay': config.WEIGHT_DECAY
    }
]

optimizer = torch.optim.AdamW(optimizer_grouped_parameters)

# Scheduler
total_steps = len(train_loader) * config.NUM_EPOCHS
warmup_steps = int(config.WARMUP_RATIO * total_steps)

if config.SCHEDULER_TYPE == 'cosine':
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )
else:
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

print(f"  Training setup:")
print(f"  Total steps: {total_steps:,}")
print(f"  Warmup steps: {warmup_steps:,}")
print(f"  Scheduler: {config.SCHEDULER_TYPE}")
print(f"  BERT LR: {config.LEARNING_RATE}")
print(f"  Classifier LR: {config.LEARNING_RATE * 10}")

In [ ]:
def train_epoch(model, dataloader, optimizer, scheduler, criterion, device):
    """Train one epoch"""
    model.train()
    
    total_loss = 0
    correct = 0
    total = 0
    
    progress = tqdm(dataloader, desc='Training')
    
    for batch in progress:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), config.MAX_GRAD_NORM)
        
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
        progress.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{correct/total:.4f}',
            'lr': f'{scheduler.get_last_lr()[0]:.2e}'
        })
    
    return total_loss / len(dataloader), correct / total


def evaluate(model, dataloader, criterion, device):
    """Evaluate model"""
    model.eval()
    
    total_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Evaluating'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            
            total_loss += loss.item()
            
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)
    
    return avg_loss, accuracy, all_preds, all_labels, all_probs


def calculate_metrics(y_true, y_pred):
    """Calculate comprehensive metrics"""
    accuracy = accuracy_score(y_true, y_pred)
    
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0
    )
    
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    macro_recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    weighted_recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    
    return {
        'accuracy': accuracy,
        'macro_f1': macro_f1,
        'macro_recall': macro_recall,
        'weighted_f1': weighted_f1,
        'weighted_recall': weighted_recall,
        'per_class': {
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'support': support
        }
    }


In [ ]:
# Training history
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'val_f1': [],
    'val_recall': [],
    'learning_rate': []
}

best_val_f1 = 0
best_val_loss = float('inf')
patience_counter = 0


print(f"Device: {config.DEVICE}")
print(f"Epochs: {config.NUM_EPOCHS}")
print(f"Batch size: {config.BATCH_SIZE}")
print(f"Training samples: {len(train_dataset):,}")

start_time = datetime.now()

for epoch in range(config.NUM_EPOCHS):
    epoch_start = datetime.now()
    
    print(f"\n{'='*70}")
    print(f" EPOCH {epoch + 1}/{config.NUM_EPOCHS}")
    print(f"{'='*70}")
    
    # Train
    train_loss, train_acc = train_epoch(
        model, train_loader, optimizer, scheduler, criterion, config.DEVICE
    )
    
    # Validate
    val_loss, val_acc, val_preds, val_labels, _ = evaluate(
        model, val_loader, criterion, config.DEVICE
    )
    
    val_metrics = calculate_metrics(val_labels, val_preds)
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_metrics['macro_f1'])
    history['val_recall'].append(val_metrics['macro_recall'])
    history['learning_rate'].append(scheduler.get_last_lr()[0])
    
    epoch_time = (datetime.now() - epoch_start).total_seconds()
    

    print(f" EPOCH {epoch + 1} RESULTS")

    print(f"Training:   Loss={train_loss:.4f}, Acc={train_acc:.4f}")
    print(f"Validation: Loss={val_loss:.4f}, Acc={val_acc:.4f}")
    print(f"\n Target Metrics:")
    print(f"  Macro F1:       {val_metrics['macro_f1']:.4f} {'' if val_metrics['macro_f1'] >= 0.75 else ''}")
    print(f"  Macro Recall:   {val_metrics['macro_recall']:.4f} {'' if val_metrics['macro_recall'] >= 0.75 else ''}")
    print(f"  Weighted F1:    {val_metrics['weighted_f1']:.4f}")
    print(f"\nTime: {epoch_time:.2f}s | LR: {scheduler.get_last_lr()[0]:.2e}")
    

    print(f"\n Per-class:")
    for i in range(config.NUM_LABELS):
        print(f"  Label {i+1} ({config.LABEL_NAMES_VI[i]:20s}): "
              f"P={val_metrics['per_class']['precision'][i]:.3f} "
              f"R={val_metrics['per_class']['recall'][i]:.3f} "
              f"F1={val_metrics['per_class']['f1'][i]:.3f}")
    


    if val_metrics['macro_f1'] > best_val_f1 + config.MIN_DELTA:
        best_val_f1 = val_metrics['macro_f1']
        best_val_loss = val_loss
        patience_counter = 0
        
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_f1': best_val_f1,
            'val_loss': best_val_loss,
        }, f'{config.OUTPUT_DIR}/{config.MODEL_SAVE_PATH}')
        
        print(f" New best F1! {best_val_f1:.4f} | Model saved")
        
        if best_val_f1 >= 0.75:
            print(f" TARGET ACHIEVED! F1 >= 75%! ")
    else:
        patience_counter += 1
        print(f"⏸  No improvement. Patience: {patience_counter}/{config.PATIENCE}")
        
        if patience_counter >= config.PATIENCE:
            print(f"\n  Early stopping after epoch {epoch + 1}")
            break

total_time = (datetime.now() - start_time).total_seconds()


print(f"Best Macro F1: {best_val_f1:.4f} {'' if best_val_f1 >= 0.75 else ''}")
print(f"Best Val Loss: {best_val_loss:.4f}")


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
epochs_range = range(1, len(history['train_loss']) + 1)


axes[0, 0].plot(epochs_range, history['train_loss'], 'o-', label='Train', linewidth=2.5, color='blue')
axes[0, 0].plot(epochs_range, history['val_loss'], 's-', label='Val', linewidth=2.5, color='red')
axes[0, 0].set_title('Loss', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)


axes[0, 1].plot(epochs_range, history['train_acc'], 'o-', label='Train', linewidth=2.5, color='blue')
axes[0, 1].plot(epochs_range, history['val_acc'], 's-', label='Val', linewidth=2.5, color='red')
axes[0, 1].set_title('Accuracy', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)


axes[0, 2].plot(epochs_range, history['val_f1'], 'D-', linewidth=2.5, color='green')
axes[0, 2].axhline(y=0.75, color='r', linestyle='--', label='Target (75%)', linewidth=2)
axes[0, 2].set_title('F1 Score ', fontsize=14, fontweight='bold')
axes[0, 2].set_xlabel('Epoch')
axes[0, 2].set_ylim(0, 1)
axes[0, 2].legend()
axes[0, 2].grid(alpha=0.3)


axes[1, 0].plot(epochs_range, history['val_recall'], '^-', linewidth=2.5, color='orange')
axes[1, 0].axhline(y=0.75, color='r', linestyle='--', label='Target (75%)', linewidth=2)
axes[1, 0].set_title('Recall ', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylim(0, 1)
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)


axes[1, 1].plot(epochs_range, history['learning_rate'], 'o-', linewidth=2.5, color='purple')
axes[1, 1].set_title('Learning Rate', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_yscale('log')
axes[1, 1].grid(alpha=0.3)

axes[1, 2].axis('off')
summary_text = f"""
Training Summary - V2
{'='*35}

Best Validation:
  • Macro F1: {best_val_f1:.4f}
  • Val Loss: {best_val_loss:.4f}

Dataset:
  • Imbalance: 2.31:1 (vs 4.55:1)
  • Label 4: 1,050 (vs 693)
  • Better balanced! 

Training:
  • Epochs: {len(history['train_loss'])}
  • Batch: {config.BATCH_SIZE}
  • LR: {config.LEARNING_RATE}
  • No Focal Loss needed!

Status: {' Success!' if best_val_f1 >= 0.75 else ' Good progress'}
"""
axes[1, 2].text(0.1, 0.5, summary_text, fontsize=11, family='monospace',
                verticalalignment='center')

plt.suptitle('Training Progress - V2 Balanced Dataset', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{config.OUTPUT_DIR}/training_history.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Load best model
print(" Loading best model...")
checkpoint = torch.load(f'{config.OUTPUT_DIR}/{config.MODEL_SAVE_PATH}')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f" Loaded model from epoch {checkpoint['epoch'] + 1}")

# Evaluate
print("\n Evaluating on test set...")
test_loss, test_acc, test_preds, test_labels, test_probs = evaluate(
    model, test_loader, criterion, config.DEVICE
)

test_metrics = calculate_metrics(test_labels, test_preds)

print("\n" + "="*70)
print(" FINAL TEST RESULTS - V2")
print("="*70)
print(f"Samples: {len(test_labels):,}")
print(f"\n Overall:")
print(f"  Accuracy:        {test_metrics['accuracy']:.4f}")
print(f"  Macro F1:        {test_metrics['macro_f1']:.4f} {'' if test_metrics['macro_f1'] >= 0.75 else ''}")
print(f"  Macro Recall:    {test_metrics['macro_recall']:.4f} {'' if test_metrics['macro_recall'] >= 0.75 else ''}")
print(f"  Weighted F1:     {test_metrics['weighted_f1']:.4f}")

if test_metrics['macro_f1'] >= 0.75 and test_metrics['macro_recall'] >= 0.75:
    print("\n EXCELLENT! Both F1 & Recall >= 75%! ")
elif test_metrics['macro_f1'] >= 0.75 or test_metrics['macro_recall'] >= 0.75:
    print("\n Very good! Target achieved!")
else:
    print("\n Good results, close to target!")

In [ ]:
# Classification report
print("\n" + "="*70)
print(" DETAILED CLASSIFICATION REPORT")
print("="*70)

label_names = [get_display_label(i, lang=config.DISPLAY_LANG) for i in range(4)]
report = classification_report(
    test_labels,
    test_preds,
    target_names=label_names,
    digits=4
)
print(report)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(test_labels, test_preds)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Counts
disp1 = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
disp1.plot(ax=axes[0], cmap='Blues', values_format='d')
axes[0].set_title('Confusion Matrix (Counts)', fontsize=14, fontweight='bold')
axes[0].grid(False)

# Normalized
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
disp2 = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=label_names)
disp2.plot(ax=axes[1], cmap='Greens', values_format='.2f')
axes[1].set_title('Confusion Matrix (Normalized)', fontsize=14, fontweight='bold')
axes[1].grid(False)

plt.tight_layout()
plt.savefig(f'{config.OUTPUT_DIR}/confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
precision = test_metrics['per_class']['precision']
recall = test_metrics['per_class']['recall']
f1 = test_metrics['per_class']['f1']

fig, ax = plt.subplots(figsize=(14, 7))

x = np.arange(len(label_names))
width = 0.25

bars1 = ax.bar(x - width, precision, width, label='Precision', 
               color='skyblue', edgecolor='black', linewidth=1.5)
bars2 = ax.bar(x, recall, width, label='Recall ', 
               color='lightcoral', edgecolor='black', linewidth=1.5)
bars3 = ax.bar(x + width, f1, width, label='F1 ', 
               color='lightgreen', edgecolor='black', linewidth=1.5)

# Add values
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xlabel('Class', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Per-Class Performance - V2 Balanced Dataset', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(label_names, rotation=15, ha='right')
ax.legend(fontsize=11)
ax.set_ylim(0, 1.1)
ax.axhline(y=0.75, color='red', linestyle='--', linewidth=2, alpha=0.5, label='Target (75%)')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{config.OUTPUT_DIR}/per_class_metrics.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Save model info
model_info = {
    'version': 'V2_Balanced',
    'model_name': config.MODEL_NAME,
    'architecture': {
        'hidden_dims': config.HIDDEN_DIMS,
        'dropout': config.DROPOUT,
        'use_batch_norm': config.USE_BATCH_NORM
    },
    'dataset': {
        'train_size': len(train_df),
        'imbalance_ratio': 2.31,
        'improvement_vs_v1': '+49%'
    },
    'training': {
        'epochs': len(history['train_loss']),
        'batch_size': config.BATCH_SIZE,
        'learning_rate': config.LEARNING_RATE,
        'focal_loss': config.USE_FOCAL_LOSS,
        'augmentation': config.USE_AUGMENTATION,
        'label_smoothing': config.LABEL_SMOOTHING
    },
    'best_metrics': {
        'val_f1': float(best_val_f1),
        'val_loss': float(best_val_loss)
    },
    'test_metrics': {
        'accuracy': float(test_metrics['accuracy']),
        'macro_f1': float(test_metrics['macro_f1']),
        'macro_recall': float(test_metrics['macro_recall']),
        'weighted_f1': float(test_metrics['weighted_f1']),
        'weighted_recall': float(test_metrics['weighted_recall'])
    },
    'label_names': {'vi': config.LABEL_NAMES_VI, 'en': config.LABEL_NAMES_EN},
    'training_date': datetime.now().isoformat(),
    'max_length': config.MAX_LENGTH
}

with open(f'{config.OUTPUT_DIR}/model_info.json', 'w', encoding='utf-8') as f:
    json.dump(model_info, f, indent=2, ensure_ascii=False)

# Save history
pd.DataFrame(history).to_csv(f'{config.OUTPUT_DIR}/training_history.csv', index=False)

# Save tokenizer
tokenizer.save_pretrained(f'{config.OUTPUT_DIR}/tokenizer')

# Save predictions
test_results = pd.DataFrame({
    'text': test_df['text_clean'].tolist(),
    'true_label': [l + 1 for l in test_labels],
    'pred_label': [p + 1 for p in test_preds],
    'correct': [t == p for t, p in zip(test_labels, test_preds)]
})
test_results.to_csv(f'{config.OUTPUT_DIR}/test_predictions.csv', index=False)

print(" All artifacts saved!")
print(f"\nLocation: {config.OUTPUT_DIR}/")

In [ ]:
# Zip
!zip -r outputs.zip {config.OUTPUT_DIR}/

# Download
from google.colab import files
files.download('outputs.zip')

print("\n Downloaded outputs.zip")

In [ ]:
def predict_eye_strain(text, model, tokenizer, device, max_length=64, display_lang=None):
    """Production inference with bilingual display support."""
    model.eval()
    display_lang = display_lang or config.DISPLAY_LANG

    text_clean = preprocess_text(text)
    if not text_clean:
        return {'error': 'Empty text'}

    encoding = tokenizer(
        text_clean,
        add_special_tokens=True,
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        logits = model(input_ids, attention_mask)
        probs = torch.softmax(logits, dim=1)
        confidence, prediction = torch.max(probs, dim=1)

    pred_idx = prediction.item()
    conf = confidence.item()
    probs_arr = probs.squeeze().cpu().numpy()

    return {
        'text': text,
        'prediction': {
            'label': pred_idx + 1,
            'severity_vi': config.LABEL_NAMES_VI[pred_idx],
            'severity_en': config.LABEL_NAMES_EN[pred_idx],
            'severity': get_display_label(label_idx=pred_idx, lang=display_lang),
            'confidence': float(conf)
        },
        'probabilities': {
            'vi': {config.LABEL_NAMES_VI[i]: float(probs_arr[i]) for i in range(config.NUM_LABELS)},
            'en': {config.LABEL_NAMES_EN[i]: float(probs_arr[i]) for i in range(config.NUM_LABELS)}
        },
        'display_lang': display_lang,
        'timestamp': datetime.now().isoformat()
    }

## Inference Demo


In [ ]:
test_cases = [
    'Mắt ổn, không vấn đề gì',
    'Hơi mỏi mắt sau làm việc',
    'Đau mắt khá nhiều, cần nghỉ',
    'Rất đau, không thể mở mắt được!!!'
]

DISPLAY_LANG = config.DISPLAY_LANG  # change to 'en' or 'vi'

print('='*70)
print('INFERENCE TEST')
print('='*70)
print(f'Display language: {DISPLAY_LANG}')

for text in test_cases:
    result = predict_eye_strain(text, model, tokenizer, config.DEVICE, display_lang=DISPLAY_LANG)
    print(f"
Input: {text}")
    print(f"Prediction: Label {result['prediction']['label']} - {result['prediction']['severity']}")
    print(f"Confidence: {result['prediction']['confidence']:.2%}")
    print('Probabilities:')
    prob_dict = result['probabilities'][DISPLAY_LANG]
    for label, prob in prob_dict.items():
        print(f'  {label}: {prob:.4f}')
    print('-' * 70)

## Notes

- Training/best-model generation follows the improved notebook.
- EN/VI display is only a post-processing layer and does not affect the trained model.
- Keep the same label order during training and inference.